In [ ]:
!pip install transformers datasets sklearn torch -q

In [4]:
!pip install evaluate

Defaulting to user installation because normal site-packages is not writeable


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:
import json
import torch
from transformers import BertTokenizerFast, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.model_selection import train_test_split
import evaluate

# ✅ 데이터 로드
with open("intent_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# ✅ 라벨 매핑
label_map = {
    "cinema_location": 0,
    "showtime": 1,
    "movie_info": 2,
    "unknown": 3
}
id_to_label_map = {v: k for k, v in label_map.items()}

# ✅ 숫자 라벨 대응 처리
processed_data = []
for d in data:
    label = d["label"]
    if isinstance(label, int):
        label = id_to_label_map.get(label, "unknown")
    processed_data.append({
        "text": d["text"],
        "label": label_map[label]
    })

# ✅ train/test split
train_data, val_data = train_test_split(data, test_size=0.2, random_state=42)
train_ds = Dataset.from_list(train_data)
val_ds = Dataset.from_list(val_data)

# ✅ 토크나이저 및 전처리
tokenizer = BertTokenizerFast.from_pretrained("beomi/kcbert-base")

def preprocess(examples):
    return tokenizer(examples["text"], truncation=True, padding='max_length', max_length=64)

train_ds = train_ds.map(preprocess, batched=True)
val_ds = val_ds.map(preprocess, batched=True)

train_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# ✅ 모델 로드
model = BertForSequenceClassification.from_pretrained("beomi/kcbert-base", num_labels=4)

# ✅ Trainer 설정
training_args = TrainingArguments(
    output_dir="./intent_model",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# ✅ 정확도 평가 함수
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return metric.compute(predictions=preds, references=labels)

# ✅ Trainer 객체 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

# ✅ 학습
trainer.train()

# ✅ 모델 저장
trainer.save_model("./intent_model")
tokenizer.save_pretrained("./intent_model")

print("✅ Intent classification model training complete!")

# ✅ movie_info 세부 분류 함수
def get_movie_info_type(text):
    if any(keyword in text for keyword in ["줄거리", "내용", "스토리"]):
        return "plot"
    elif any(keyword in text for keyword in ["장르", "카테고리", "종류"]):
        return "genre"
    elif any(keyword in text for keyword in ["평점", "점수", "몇 점"]):
        return "rating"
    elif any(keyword in text for keyword in ["감독", "출연", "배우", "감독이 누구"]):
        return "staff"
    else:
        return "unknown"

# ✅ 예측 및 후처리
from torch.nn.functional import softmax

# 예시 문장
example = "어벤져스 줄거리 알려줘"

# 입력 인코딩
inputs = tokenizer(example, return_tensors="pt")

# 모델 예측
model.eval()
with torch.no_grad():
    outputs = model(**inputs)
    probs = softmax(outputs.logits, dim=1)
    label_id = probs.argmax(dim=1).item()

# 라벨 해석
id2label = {v: k for k, v in label_map.items()}
intent = id2label[label_id]
print("🎯 예측된 의도:", intent)

# 세부 분류 (movie_info인 경우)
if intent == "movie_info":
    detail = get_movie_info_type(example)
    print("🔍 movie_info 세부 타입:", detail)

Map:   0%|          | 0/192 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Accuracy
1,0.646800,0.166694,0.979167
2,0.040900,0.006080,1.000000
3,0.004900,0.003754,1.000000


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ Intent classification model training complete!
🎯 예측된 의도: movie_info
🔍 movie_info 세부 타입: plot


In [6]:
# 학습 모델 테스트
from transformers import BertTokenizerFast, BertForSequenceClassification
import torch

# ✅ 모델과 토크나이저 로드
model = BertForSequenceClassification.from_pretrained("./intent_model")
tokenizer = BertTokenizerFast.from_pretrained("./intent_model")

# 라벨 맵 정의 (학습 때와 동일하게)
id_to_label_map = {
    0: "cinema_location",
    1: "showtime",
    2: "movie_info",
    3: "unknown"
}

# movie_info 세부 분류
def get_movie_info_type(text):
    if any(k in text for k in ["줄거리", "내용", "스토리"]):
        return "plot"
    elif any(k in text for k in ["장르", "카테고리", "종류"]):
        return "genre"
    elif any(k in text for k in ["평점", "점수", "몇 점"]):
        return "rating"
    elif any(k in text for k in ["감독", "출연", "배우", "감독이 누구"]):
        return "staff"
    else:
        return "unknown"

# 테스트 예시
def test_intent(text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        logits = model(**inputs).logits
    label_id = logits.argmax().item()
    intent = id_to_label_map[label_id]
    print(f"📌 입력: {text}")
    print(f"🧠 예측된 의도: {intent}")

    if intent == "movie_info":
        detail = get_movie_info_type(text)
        print(f"🔍 movie_info 세부 요청: {detail}")

# ✅ 예시 실행
test_intent("해운대 줄거리 알려줘")
test_intent("강남 롯데시네마 주소 알려줘")
test_intent("광명시 영화관 어디야?")
test_intent("강남 메가박스에서 슈퍼맨 상영시간 보여줘")
test_intent("슈퍼맨 감독 알려줘")
test_intent("슈퍼맨 감독이 누구야?")
test_intent("슈퍼맨 감독은?")

📌 입력: 해운대 줄거리 알려줘
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: plot
📌 입력: 강남 롯데시네마 주소 알려줘
🧠 예측된 의도: cinema_location
📌 입력: 광명시 영화관 어디야?
🧠 예측된 의도: cinema_location
📌 입력: 강남 메가박스에서 슈퍼맨 상영시간 보여줘
🧠 예측된 의도: showtime
📌 입력: 슈퍼맨 감독 알려줘
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff
📌 입력: 슈퍼맨 감독이 누구야?
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff
📌 입력: 슈퍼맨 감독은?
🧠 예측된 의도: movie_info
🔍 movie_info 세부 요청: staff


In [14]:
!pip install seqeval

Defaulting to user installation because normal site-packages is not writeable
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16183 sha256=70bee3a7f491daae76926facd94da7b9440acaa1eee2a10e57f9e0f0c91b975b
  Stored in directory: c:\users\user\appdata\local\pip\cache\wheels\5f\b8\73\0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [ ]:
# NER 학습 모델 저장
import torch
from transformers import BertTokenizerFast, BertForTokenClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split
import numpy as np
from datasets import Dataset

# 1) id2label / label2id 정의 (NER 태그)
id2label = {
    0: "O",
    1: "B-REGION",
    2: "I-REGION",
    3: "B-CINEMA",
    4: "I-CINEMA",
    5: "B-MOVIE",
    6: "I-MOVIE",
}
label2id = {v: k for k, v in id2label.items()}

# 2) 준비한 json 데이터 불러오기 (예: 'ner_dataset.json')
import json
with open("ner_training_data_from_megabox.json", "r", encoding="utf-8") as f:
    ner_data = json.load(f)

# 3) Dataset 객체 생성
dataset = Dataset.from_list(ner_data)

# 4) 토크나이저 로드
tokenizer = BertTokenizerFast.from_pretrained("beomi/kcbert-base")

# 5) 토큰화 및 라벨 정렬 함수
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], 
        is_split_into_words=True, 
        truncation=True, 
        padding="max_length", 
        max_length=64
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)  # ignore index for loss calculation
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # B- 태그가 I- 태그로 변경 (optional)
                if label[word_idx] % 2 == 1:  # odd index means B-
                    label_ids.append(label[word_idx] + 1)
                else:
                    label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 6) 데이터셋에 토큰화 적용
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# 7) train / validation split
train_size = int(0.8 * len(tokenized_dataset))
train_dataset = tokenized_dataset.select(range(train_size))
eval_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

# 8) 데이터 포맷 지정
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 9) 모델 로드 (num_labels = 태그 개수)
model = BertForTokenClassification.from_pretrained(
    "beomi/kcbert-base", 
    num_labels=len(id2label), 
    id2label=id2label, 
    label2id=label2id
)

# 10) 평가 지표 정의
import evaluate

metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    # 라벨 -100 제외하고 mapping
    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 11) TrainingArguments 설정
training_args = TrainingArguments(
    output_dir="./ner_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 12) Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# 13) 학습 실행
trainer.train()

# 14) 모델 및 토크나이저 저장
trainer.save_model("./ner_model")
tokenizer.save_pretrained("./ner_model")

print("✅ NER model training complete!")

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at beomi/kcbert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.023700,0.006904,0.967822,0.984887,0.976280,0.997472
2,0.000500,0.000268,1.000000,1.000000,1.000000,1.000000
3,0.000400,0.000066,1.000000,1.000000,1.000000,1.000000
4,0.000100,0.000050,1.000000,1.000000,1.000000,1.000000
5,0.000100,0.000046,1.000000,1.000000,1.000000,1.000000


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ NER model training complete!


In [ ]:
# 데이터셋 만들기
import random
import json

regions = ["서울", "부산", "인천", "대구", "광주", "대전", "수원", "강남", "홍대", "강동", "광명"]
cinemas = ["CGV", "메가박스", "롯데시네마", "영화관"]
movies = ["기생충", "아바타", "슈퍼맨", "스파이더맨", "타이타닉", "아이언맨", "어벤져스", "전지적 독자 시점", "킹 오브 킹스"]

templates = [
    ["{region}", "{cinema}", "{movie}", "몇", "시", "해"],
    ["{region}", "{cinema}", "{movie}", "줄거리", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "상영시간", "알려줘"],
    ["{region}", "{cinema}", "{movie}", "감독", "누구야"],
    ["{region}", "{cinema}", "위치", "알려줘"],
    ["{movie}", "평점", "알려줘"],
    ["{region}", "{cinema}", "주소", "알려줘"],
    ["{movie}", "장르", "알려줘"]
    ["{region}", "{cinema}", "위치", "찾아줘"],
    ["{movie}", "평점", "찾아줘"],
    ["{region}", "{cinema}", "주소", "찾아줘"],
    ["{movie}", "장르", "찾아줘"]
    ["{region}", "{cinema}", "위치", "뭐야"],
    ["{movie}", "평점", "뭐야"],
    ["{region}", "{cinema}", "주소", "뭐야"],
    ["{movie}", "장르", "뭐야"]
]

label_map = {
    "O": 0,
    "B-REGION": 1,
    "I-REGION": 2,
    "B-CINEMA": 3,
    "I-CINEMA": 4,
    "B-MOVIE": 5,
    "I-MOVIE": 6
}

def label_tokens(tokens, region, cinema, movie):
    labels = []
    for token in tokens:
        if token == region:
            labels.append(label_map["B-REGION"])
        elif token == cinema:
            labels.append(label_map["B-CINEMA"])
        elif token == movie:
            labels.append(label_map["B-MOVIE"])
        else:
            labels.append(label_map["O"])
    return labels

data = []
for _ in range(1000):
    region = random.choice(regions)
    cinema = random.choice(cinemas)
    movie = random.choice(movies)
    template = random.choice(templates)
    tokens = [t.format(region=region, cinema=cinema, movie=movie) for t in template]
    ner_tags = label_tokens(tokens, region, cinema, movie)
    data.append({"tokens": tokens, "ner_tags": ner_tags})

with open("ner_training_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=4)

In [14]:
import json
import random
import re

# JSON 경로는 raw string으로 처리
with open(r"C:\kosmo\test_data\IH\all_cinema_with_showtimes.json", "r", encoding="utf-8") as f:
    cinema_data = json.load(f)

region_set = set()
for item in cinema_data:
    region = item.get("region", [])
    if isinstance(region, list):
        region_set.update(region)
    else:
        region_set.add(region)

regions = list(region_set)
cinemas = ["롯데시네마", "영화관", "메가박스"]

movies = [
    "판타스틱 4: 새로운 출발", "전지적 독자 시점", "명탐정 코난: 척안의 잔상", "킹 오브 킹스", "노이즈", "F1 더 무비",
    "베베핀 극장판: 사라진 베베핀과 핑크퐁 대모험", "쥬라기 월드: 새로운 시작", "슈퍼맨", "이사", "괴기열차", "메간 2.0",
    "망국전쟁 : 뉴라이트의 시작", "드래곤 길들이기", "스왈로우테일 버터플라이", "킹 오브 프리즘", "커미션", "엘리오",
    "미세리코르디아", "8과 1/2", "바람계곡의 나우시카", "진격의 거인 완결편 더 라스트 어택", "우리들의 교복시절", "여름이 지나가면",
    "봄밤", "네이키드 런치", "바다호랑이", "일과 날", "발코니의 여자들", "라이언 일병 구하기", "비밀의 화원", "해피엔드",
    "오키나와 블루노트", "그을린 사랑", "신명", "미션 임파서블: 파이널 레코닝", "하이파이브",
    "극장판 프로젝트 세카이 부서진 세카이와 전해지지 않는 미쿠의 노래", "28년 후"
]

templates = [
"{region}에 {cinema} 어디 있어?",
    "{region} {cinema} 위치 알려줘",
    "{region} {cinema} 주소 알려줄래?",
    "{region}에 있는 {cinema} 어디야?",
    "{cinema} {region}에 있나요?",
    "{cinema}가 {region}에 있어?",
    "{region} {cinema} 찾아줘",
    "{region}에서 {cinema} 어디 있지?",
    "{region} {cinema}가 어디야?",
    "{region} {cinema}가 있나요?",
    "{region} {cinema} 주소가 뭐야?",
    "{region} {cinema} 정확한 위치 알려줘",
    "{region} {cinema} 상세 위치 알려줘",
    "{region} {cinema} 위치 궁금해",
    "{region}에 {cinema} 위치 어디야?",
    "{region}에 {movie} 언제 해?",
    "{region}에서 {movie} 상영시간 알려줘",
    "{region} {movie} 오늘 몇 시에 해?",
    "{region}에 있는 영화 {movie} 상영시간 궁금해",
    "{region} {movie} 상영시간 확인해줘",
    "{movie}가 {region}에서 언제 상영돼?",
    "{movie} {region}에 상영 시간 알려줘",
    "{region}에서 {movie} 상영 시간 알려주세요",
    "{region} {movie} 상영 시간 알려줄래?",
    "{region} {movie} 시간대 알려줘",
    "{region} {movie} 언제 볼 수 있어?",
    "{region}에서 {movie} 언제 하는지 알려줘",
    "{region} {movie} 오늘 상영 시간 알려줘",
    "{region} {movie} 지금 상영하나요?",
    "{region}에서 {movie} 시간 좀 알려줘",
    "{region}에 있는 {movie} 오늘 상영 시간 뭐야?",
    "{cinema} 어디야?",
    "{cinema} 위치 알려줘",
    "{cinema} 주소 알려줘",
    "{cinema} 어디에 있나요?",
    "{cinema} 위치가 어디야?",
    "{cinema} 위치 궁금해",
    "{cinema} 위치 알려줄래?",
    "{cinema} 정확한 주소 알려줘",
    "{cinema} 상세 위치 알려줘",
    "{cinema} 어디 위치해?",
    "{cinema} 위치 좀 알려줘",
    "{cinema} 주소가 어떻게 돼?",
    "{cinema} 위치를 알고 싶어",
    "{cinema}에서 {movie} 언제 해?",
    "{cinema} {movie} 상영 시간 알려줘",
    "{cinema}에서 {movie} 오늘 상영 시간?",
    "{cinema} {movie} 시간대 알려줘",
    "{cinema}에서 {movie} 상영 중이야?",
    "{cinema} {movie} 언제 볼 수 있지?",
    "{cinema}에서 {movie} 상영 시간 확인해줘",
    "{cinema}에서 {movie} 오늘 몇 시에 해?",
    "{region}에 있는 {cinema} 어디야?",
    "{region} {cinema} 있나?",
    "{region}에 {cinema} 있지?",
    "{region} {movie} 언제 해?",
    "{movie} {region}에서 볼 수 있어?",
    "{movie} 오늘 상영시간 좀 알려줘",
    "{cinema} 어디에 있어?",
    "{cinema} 위치 어디야?"
]

label_map = {
    "O": 0,
    "B-REGION": 1,
    "I-REGION": 2,
    "B-CINEMA": 3,
    "I-CINEMA": 4,
    "B-MOVIE": 5,
    "I-MOVIE": 6
}


def custom_tokenize(sentence, region, cinema, movie):
    # 개체 먼저 치환 (구분자 삽입)
    sentence = sentence.replace(region, f" [REGION]{region}[/REGION] ")
    sentence = sentence.replace(cinema, f" [CINEMA]{cinema}[/CINEMA] ")
    sentence = sentence.replace(movie, f" [MOVIE]{movie}[/MOVIE] ")

    tokens = []
    ner_tags = []

    # 정규식: 개체명 태그 포함 덩어리, 혹은 단어/숫자, 구두점 분리
    pattern = re.compile(r'\[REGION\].+?\[/REGION\]|\[CINEMA\].+?\[/CINEMA\]|\[MOVIE\].+?\[/MOVIE\]|\w+|[^\w\s]')
    parts = pattern.findall(sentence)

    for part in parts:
        if part.startswith("[REGION]"):
            word = part.replace("[REGION]", "").replace("[/REGION]", "")
            for i, w in enumerate(word.split()):
                tokens.append(w)
                ner_tags.append(label_map["B-REGION"] if i == 0 else label_map["I-REGION"])
        elif part.startswith("[CINEMA]"):
            word = part.replace("[CINEMA]", "").replace("[/CINEMA]", "")
            for i, w in enumerate(word.split()):
                tokens.append(w)
                ner_tags.append(label_map["B-CINEMA"] if i == 0 else label_map["I-CINEMA"])
        elif part.startswith("[MOVIE]"):
            word = part.replace("[MOVIE]", "").replace("[/MOVIE]", "")
            for i, w in enumerate(word.split()):
                tokens.append(w)
                ner_tags.append(label_map["B-MOVIE"] if i == 0 else label_map["I-MOVIE"])
        else:
            tokens.append(part)
            ner_tags.append(label_map["O"])

    return tokens, ner_tags

data = []

for _ in range(5000):
    region = random.choice(regions)
    cinema = random.choice(cinemas)
    movie = random.choice(movies)
    template = random.choice(templates)

    sentence = template.format(region=region, cinema=cinema, movie=movie)
    tokens, ner_tags = custom_tokenize(sentence, region, cinema, movie)

    data.append({
        "tokens": tokens,
        "ner_tags": ner_tags
    })

with open("ner_training_data_v2.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print("✅ NER 학습 데이터 저장 완료")


✅ NER 학습 데이터 저장 완료


In [ ]:
import json
import numpy as np
from datasets import Dataset
from transformers import (
    BertTokenizerFast,
    BertForTokenClassification,
    Trainer,
    TrainingArguments
)
import evaluate

# 1) NER 태그 id2label, label2id
id2label = {
    0: "O",
    1: "B-REGION",
    2: "I-REGION",
    3: "B-CINEMA",
    4: "I-CINEMA",
    5: "B-MOVIE",
    6: "I-MOVIE",
}
label2id = {v: k for k, v in id2label.items()}

# 2) 기존 학습된 모델 경로
model_path = "C:\\kosmo\\model\\ner_model"

# 3) 토크나이저, 모델 불러오기
tokenizer = BertTokenizerFast.from_pretrained(model_path)
model = BertForTokenClassification.from_pretrained(
    model_path,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id,
)

# 4) 추가 학습용 JSON 데이터 로드
with open("ner_training_data.json", "r", encoding="utf-8") as f1, \
     open("ner_training_data_v2.json", "r", encoding="utf-8") as f2:
    data1 = json.load(f1)
    data2 = json.load(f2)

# 두 데이터 합치기
combined_data = data1 + data2

# Dataset 생성
dataset = Dataset.from_list(combined_data)

# 5) 토큰화 및 라벨 정렬 함수
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        is_split_into_words=True,
        truncation=True,
        padding="max_length",
        max_length=64,
    )
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                # B- 태그가 I- 태그로 바뀌도록 (optional)
                if label[word_idx] % 2 == 1:
                    label_ids.append(label[word_idx] + 1)
                else:
                    label_ids.append(label[word_idx])
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# 6) 토큰화 적용
tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

# 7) train/validation split
train_size = int(0.8 * len(tokenized_dataset))
train_dataset = tokenized_dataset.select(range(train_size))
eval_dataset = tokenized_dataset.select(range(train_size, len(tokenized_dataset)))

# 8) 포맷 설정
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
eval_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# 9) 평가 지표 정의 (seqeval)
metric = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    preds = np.argmax(predictions, axis=2)

    true_labels = [
        [id2label[l] for l in label if l != -100]
        for label in labels
    ]
    true_preds = [
        [id2label[p] for (p, l) in zip(pred, label) if l != -100]
        for pred, label in zip(preds, labels)
    ]

    results = metric.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 10) Trainer용 TrainingArguments 설정
training_args = TrainingArguments(
    output_dir=model_path,   # 기존 모델 덮어쓰기. 새로 저장하려면 다른 경로 지정
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
)

# 11) Trainer 생성
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics,
)

# 12) 학습 시작
trainer.train()

# 13) 학습된 모델 및 토크나이저 저장
trainer.save_model("./ner_model_v3")
tokenizer.save_pretrained("./ner_model_v3")

print("✅ NER 모델 추가 학습 및 저장 완료!")


Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.000200,0.000023,1.000000,1.000000,1.000000,1.000000
2,0.000000,0.000010,1.000000,1.000000,1.000000,1.000000
3,0.000000,0.000009,1.000000,1.000000,1.000000,1.000000


C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
C:\Users\user\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


✅ NER 모델 추가 학습 및 저장 완료!
